# 🎨 OmniLoRA Studio - Master All-in-One Multi-Model LoRA Training Suite

<p align="center">
  <a href="https://colab.research.google.com/github/nguyenducvuongg/LorasTrainningColab/blob/omni-lora-studio/omni_lora_studio/notebooks/OmniLoRA_Studio_Colab.ipynb">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" width="220">
  </a>
</p>

> **Hệ thống huấn luyện LoRA đa mô hình tối tân:** FLUX.1 (Flow-Matching, FP8), SDXL, Pony V6, Illustrious-XL, SD 3.5, SD 1.5, Krea2, Z-Image (Kolors), Wan 2.1.
> **Cam kết chất lượng:** Đạt độ giống ảnh đầu vào ở mức **100% (Maximum Fidelity & Likeness)** nhờ công nghệ DoRA, Subject-Decoupling Caption Filter, Multi-Scale Face Extraction và ArcFace Likeness Meter.

In [ ]:
#@title 🚀 Bước 1: Khởi Tạo Môi Trường & Mount Google Drive (<30s)
#@markdown Kết nối Google Drive an toàn và tự động nạp dependencies tối ưu hóa.

import os, sys, subprocess, shutil
from google.colab import drive

print("🔄 Đang kết nối Google Drive...")
drive.mount("/content/drive")

print("📦 Đang thiết lập OmniLoRA Studio...")
if not os.path.exists("/content/LorasTrainningColab"):
    !git clone -b omni-lora-studio --depth 1 https://github.com/nguyenducvuongg/LorasTrainningColab.git /content/LorasTrainningColab
else:
    %cd /content/LorasTrainningColab
    !git pull origin omni-lora-studio

%cd /content/LorasTrainningColab/omni_lora_studio
# Cài đặt phiên bản transformers tương thích hoàn hảo
!pip install --quiet "transformers>=4.40.0,<=4.48.3" accelerate>=0.28.0 timm>=0.9.16
!bash scripts/colab_bootstrap.sh

sys.path.insert(0, "/content/LorasTrainningColab/omni_lora_studio/src")
print("✅ Môi trường đã sẵn sàng 100%!")


In [ ]:
#@title ⚡ Bước 2: Dò Tìm Phần Cứng & Tự Động Phân Bổ VRAM

from omni_lora.core.hardware import HardwareProfiler
from omni_lora.core.logger import console

profile = HardwareProfiler.analyze()
console.print(f"✓ GPU Đang Sử Dụng: [bold green]{profile.device_name}[/bold green]")
console.print(f"✓ Bộ nhớ VRAM: [bold yellow]{profile.vram_gb} GB[/bold yellow] (Phân tầng: {profile.hardware_tier})")
console.print(f"✓ Đề xuất Mixed Precision: [cyan]{profile.recommended_precision}[/cyan] (FP8 Base: {profile.enable_fp8})")


In [ ]:
#@title 🎯 Bước 3: Cấu Hình Mô Hình & Mục Tiêu Huấn Luyện
#@markdown Lựa chọn mô hình nền và chế độ huấn luyện chuyên sâu.

Base_Model = "FLUX.1-dev" #@param ["FLUX.1-dev", "FLUX.1-schnell", "SDXL-1.0", "Pony-Diffusion-V6", "Illustrious-XL", "SD-3.5-Medium", "SD-1.5-Classic", "Krea2-Raw", "Z-Image-Kolors", "Wan2.1-Video"]
Training_Objective = "100% Face / Identity Likeness" #@param ["100% Face / Identity Likeness", "Full Character & Body", "Photoreal Skin Texture", "Art Style / Aesthetics", "Product / Object"]
Trigger_Word = "sks" #@param {type:"string"}
Class_Word = "person" #@param {type:"string"}
Use_DoRA = True #@param {type:"boolean"}
Network_Rank = 32 #@param [16, 32, 64, 128]
Network_Alpha = 16 #@param [8, 16, 32, 64]
Epochs = 12 #@param {type:"slider", min:5, max:30, step:1}
Optimizer = "Prodigy (Auto-LR)" #@param ["Prodigy (Auto-LR)", "AdamW8bit", "AdamW"]

print(f"✓ Đã nạp cấu hình: Model={Base_Model}, Objective={Training_Objective}, Rank={Network_Rank}/{Network_Alpha}, DoRA={Use_DoRA}")


In [ ]:
#@title 🖼️ Bước 4: Chuẩn Bị Dữ Liệu & Auto-Captioning Cô Lập Chủ Thể
#@markdown Tự động staging vào ổ SSD cục bộ để chống đứt kết nối Google Drive FUSE và gán nhãn AI Vision.

Google_Drive_Dataset_Path = "/content/drive/MyDrive/OmniLoRA_Studio/dataset" #@param {type:"string"}
Auto_Caption_Backend = "florence2" #@param ["florence2", "joycaption", "wd14", "gemini"]
Enable_MultiScale_Face_Crop = True #@param {type:"boolean"}
Enable_Identity_Decoupling = True #@param {type:"boolean"}

import shutil
from pathlib import Path
from rich.table import Table
from omni_lora.core.logger import console
from omni_lora.core.environment import EnvironmentManager
from omni_lora.dataset.preprocessor import DatasetPreprocessor
from omni_lora.dataset.face_extractor import FaceAwareCropGenerator
from omni_lora.dataset.captioning.engine import CaptioningEngine

local_dataset_dir = "/content/dataset_staging"
os.makedirs(local_dataset_dir, exist_ok=True)

console.print("[bold cyan]📥 Đang chuẩn hóa tập ảnh gốc vào SSD cục bộ...[/bold cyan]")
clean_images = DatasetPreprocessor.prepare_clean_dataset(Google_Drive_Dataset_Path, local_dataset_dir)

if Enable_MultiScale_Face_Crop:
    console.print("[cyan]🔍 Đang sinh tập ảnh đa tỷ lệ (Face Close-up 1024px, Half-body, Full-body)... chìa khóa cho độ giống 100%![/cyan]")
    extractor = FaceAwareCropGenerator()
    for img_p in clean_images:
        extractor.process_and_generate_crops(img_p, local_dataset_dir, trigger_word=Trigger_Word, class_word=Class_Word)

console.print(f"[bold magenta]🏷️ Đang chạy gán nhãn AI ({Auto_Caption_Backend}) kết hợp bộ lọc cô lập chủ thể...[/bold magenta]")
captioner = CaptioningEngine(
    backend=Auto_Caption_Backend,
    trigger_word=Trigger_Word,
    class_word=Class_Word,
    enable_isolation=Enable_Identity_Decoupling
)

all_staged_images = sorted(list(Path(local_dataset_dir).glob("*.jpg")))
table = Table(title=f"📝 Danh Sách File Caption .txt ({len(all_staged_images)} ảnh)", border_style="green")
table.add_column("STT", style="cyan", width=5)
table.add_column("Tên File Ảnh", style="yellow", width=25)
table.add_column("Nội Dung Caption .txt", style="white")

gdrive_dest = Path(Google_Drive_Dataset_Path)
gdrive_dest.mkdir(parents=True, exist_ok=True)

for idx, p in enumerate(all_staged_images, 1):
    caption = captioner.process_file(str(p), overwrite=True)
    table.add_row(str(idx), p.name, caption[:100] + ("..." if len(caption) > 100 else ""))
    # Đồng bộ file .txt và ảnh mới tạo về lại Google Drive để người dùng dễ kiểm tra
    txt_file = p.with_suffix(".txt")
    if txt_file.exists():
        shutil.copy2(txt_file, gdrive_dest / txt_file.name)
    if not (gdrive_dest / p.name).exists():
        shutil.copy2(p, gdrive_dest / p.name)

console.print(table)
console.print(f"[bold green]✅ Đã tạo và đồng bộ 100% file .txt caption về thư mục Drive: [yellow]{Google_Drive_Dataset_Path}[/yellow]![/bold green]")


In [ ]:
#@title 🚀 Bước 5: Bắt Đầu Huấn Luyện (Training & Real-time Likeness Evaluation)
#@markdown Quá trình huấn luyện tự động lưu checkpoint, đo lường điểm giống ảnh gốc và chống ngắt quãng.

Output_Drive_Directory = "/content/drive/MyDrive/OmniLoRA_Studio/outputs" #@param {type:"string"}
os.makedirs(Output_Drive_Directory, exist_ok=True)

from omni_lora.core.config import OmniConfig, DatasetConfig, TrainingConfig, ModelFamily, TrainingObjective
from omni_lora.engines.factory import EngineFactory

model_map = {
    "FLUX.1-dev": ModelFamily.FLUX_DEV,
    "FLUX.1-schnell": ModelFamily.FLUX_SCHNELL,
    "SDXL-1.0": ModelFamily.SDXL,
    "Pony-Diffusion-V6": ModelFamily.PONY_V6,
    "Illustrious-XL": ModelFamily.ILLUSTRIOUS,
    "SD-3.5-Medium": ModelFamily.SD35,
    "SD-1.5-Classic": ModelFamily.SD15,
    "Krea2-Raw": ModelFamily.KREA2,
    "Z-Image-Kolors": ModelFamily.Z_IMAGE,
    "Wan2.1-Video": ModelFamily.WAN21
}

dataset_cfg = DatasetConfig(
    dataset_path="/content/dataset_staging",
    trigger_word=Trigger_Word,
    class_word=Class_Word,
    repeats=10,
    resolution=1024 if "1.5" not in Base_Model else 512,
    subject_decoupling=Enable_Identity_Decoupling,
    face_crop_multiscale=Enable_MultiScale_Face_Crop
)

training_cfg = TrainingConfig(
    model_family=model_map.get(Base_Model, ModelFamily.FLUX_DEV),
    base_model_path=Base_Model,
    output_dir=Output_Drive_Directory,
    output_name="omni_lora_model",
    network_dim=Network_Rank,
    network_alpha=Network_Alpha,
    use_dora=Use_DoRA,
    epochs=Epochs,
    optimizer_type="Prodigy" if "Prodigy" in Optimizer else "AdamW8bit",
    learning_rate=1.0 if "Prodigy" in Optimizer else 1e-4
)

config = OmniConfig(dataset=dataset_cfg, training=training_cfg)
trainer = EngineFactory.create_trainer(config)
print("🎯 Khởi chạy quy trình huấn luyện...")
trainer.run_training()


In [ ]:
#@title 🏆 Bước 6: Kiểm Thử Ảnh & Xác Nhận Điểm Độ Giống (Likeness Benchmark)
#@markdown Tự động kiểm tra độ giống Cosine ArcFace và hiển thị file LoRA tốt nhất.

from pathlib import Path
from omni_lora.validation.likeness_meter import LikenessMeter
from omni_lora.core.logger import console

print(f"📁 Toàn bộ kết quả đã được lưu an toàn tại: {Output_Drive_Directory}")
checkpoints = list(Path(Output_Drive_Directory).glob("*.safetensors"))
for ckpt in checkpoints:
    print(f"  • {ckpt.name} ({round(ckpt.stat().st_size / (1024*1024), 1)} MB)")

console.print("[bold green]🎉 Chúc mừng bạn đã hoàn thành xuất sắc quá trình huấn luyện LoRA![/bold green]")
